### Imports

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
import os
import datetime
import matplotlib.pyplot as plt
import datetime
import pandas as pd 
from torch.nn import init
from kaggledatahandler import KaggleDataHandler
from audiopreprocessing import SoundDS
from sklearn.metrics import confusion_matrix


### Creating dataset (Training & Test)

In [5]:
KDHandler = KaggleDataHandler()
datasets_filepath_organized, y_datasets_filepath_organized = KDHandler.create_set()
print("Created the following filepaths with correspondong label\n:", datasets_filepath_organized.keys())

Created the following filepaths with correspondong label
: dict_keys(['fold1', 'fold2', 'fold3', 'fold4', 'fold5', 'fold6', 'fold7', 'fold8', 'fold9', 'fold10'])


In [6]:

preprocessed_datasets = {}
audiopreprocessor = SoundDS()

for fold in datasets_filepath_organized:
    new_prepro_fold = []
    for i, file in enumerate(datasets_filepath_organized[fold]):
        class_ID = y_datasets_filepath_organized[fold][i]
        filepath = file
        spectrgram, class_id = audiopreprocessor.__getitem__(filepath, class_ID)
        preprocessed_wav = (spectrgram, class_id)
        new_prepro_fold.append(preprocessed_wav)
    print("Finished", fold)
    preprocessed_datasets[fold] = new_prepro_fold

Finished fold1
Finished fold2
Finished fold3
Finished fold4
Finished fold5
Finished fold6
Finished fold7
Finished fold8
Finished fold9
Finished fold10


### CNN Model

In [7]:
import torch.nn.functional as F
import torch
import torch.nn as nn
from torch.nn import init

class AudioClassifier(nn.Module):

    def __init__(self):
        super().__init__()
        conv_layers = []

        # First Convolution Block
        self.conv1 = nn.Conv2d(2, 32, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
        self.relu1 = nn.ReLU()
        self.bn1 = nn.BatchNorm2d(32)
        init.kaiming_normal_(self.conv1.weight, nonlinearity='relu')
        self.conv1.bias.data.zero_()
        conv_layers += [self.conv1, self.relu1, self.bn1]

        # Second Convolution Block
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        self.relu2 = nn.ReLU()
        self.bn2 = nn.BatchNorm2d(64)
        init.kaiming_normal_(self.conv2.weight, nonlinearity='relu')
        self.conv2.bias.data.zero_()
        conv_layers += [self.conv2, self.relu2, self.bn2]

        # Third Convolution Block
        self.conv3 = nn.Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        self.relu3 = nn.ReLU()
        self.bn3 = nn.BatchNorm2d(128)
        init.kaiming_normal_(self.conv3.weight, nonlinearity='relu')
        self.conv3.bias.data.zero_()
        conv_layers += [self.conv3, self.relu3, self.bn3]

        # Fourth Convolution Block
        self.conv4 = nn.Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        self.relu4 = nn.ReLU()
        self.bn4 = nn.BatchNorm2d(256)
        init.kaiming_normal_(self.conv4.weight, nonlinearity='relu')
        self.conv4.bias.data.zero_()
        conv_layers += [self.conv4, self.relu4, self.bn4]

        # Fifth Convolution Block
        self.conv5 = nn.Conv2d(256, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        self.relu5 = nn.ReLU()
        self.bn5 = nn.BatchNorm2d(512)
        init.kaiming_normal_(self.conv5.weight, nonlinearity='relu')
        self.conv5.bias.data.zero_()
        conv_layers += [self.conv5, self.relu5, self.bn5]

        # Convolution blocks wrapped in Sequential
        self.conv = nn.Sequential(*conv_layers)

        # Final global Adaptive Pooling
        self.ap = nn.AdaptiveAvgPool2d(output_size=1)

        self.dropout = nn.Dropout(p=0.5)
        self.lin = nn.Linear(512, 10)

    def forward(self, x):
        x = self.conv(x)     # Goes through conv + relu + bn 5 times
        x = self.ap(x)     
        x = x.view(x.shape[0], -1)
        x = self.dropout(x)
        x = self.lin(x)
        return x


### Testing loop

In [8]:

def test(model, val_dl):
  correct_prediction = 0
  total_prediction = 0

  all_labels = []
  all_preds = []

  # Disable gradient updates
  with torch.no_grad():
    for data in val_dl:
      # Get the input features and target labels, and put them on the GPU
      inputs, labels = data[0].to(device), data[1].to(device)

      # Normalize the inputs
      inputs_m, inputs_s = inputs.mean(), inputs.std()
      inputs = (inputs - inputs_m) / inputs_s

      # Get predictions
      outputs = model(inputs)

      # Get the predicted class with the highest score
      _, prediction = torch.max(outputs,1)
      
      all_labels.append(labels.cpu())
      all_preds.append(prediction.cpu())
      # Count of predictions that matched the target label
      correct_prediction += (prediction == labels).sum().item()
      total_prediction += prediction.shape[0]
    
  acc = correct_prediction/total_prediction
  all_labels = all_labels
  all_preds = torch.cat(all_preds)
  print(f'Accuracy: {acc:.2f}, Total items: {total_prediction}\n')
  return acc, all_labels, all_preds


### Training loop

In [9]:
def training(model, train_dl, num_epochs, device):
  # Loss Function, Optimizer and Scheduler
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adamax(model.parameters(),lr=0.001)
  scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.2,
                                                steps_per_epoch=int(len(train_dl)),
                                                epochs=num_epochs,
                                                anneal_strategy='cos')
  
  losses = []
  accuracies = []
  lrs = []

  # Repeat for each epoch
  for epoch in range(num_epochs):
    running_loss = 0.0
    correct_prediction = 0
    total_prediction = 0
    lr = []

    # Repeat for each batch in the training set
    for i, data in enumerate(train_dl):
        # Get the input features and target labels, and put them on the GPU
        inputs, labels = data[0].to(device), data[1].to(device)

        # Normalize the inputs
        inputs_m, inputs_s = inputs.mean(), inputs.std()
        inputs = (inputs - inputs_m) / inputs_s

        # Zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        # Keep stats for Loss and Accuracy
        running_loss += loss.item()

        # Current learning rate
        current_lr = optimizer.param_groups[0]["lr"]
        lr.append(current_lr)

        # Get the predicted class with the highest score
        _, prediction = torch.max(outputs,1)
        # Count of predictions that matched the target label
        correct_prediction += (prediction == labels).sum().item()
        total_prediction += prediction.shape[0]

        #if i % 10 == 0:    # print every 10 mini-batches
        #    print('[%d, %5d] loss: %.3f' % (epoch + 1, i + 1, running_loss / 10))
    
    # Print stats at the end of the epoch
    num_batches = len(train_dl)
    avg_loss = running_loss / num_batches
    acc = correct_prediction/total_prediction
    losses.append(avg_loss)
    accuracies.append(acc)
    lrs.append(sum(lr)/len(lr))

    print(f'Epoch: {epoch}, Loss: {avg_loss:.2f}, Accuracy: {acc:.2f}')

  print('Finished Training\n')
  return losses, accuracies, lrs

### Ploting function

In [10]:

def plot_and_save_cv_curves(
    train_accuracies,
    train_losses,
    test_accuracies,
    test_accuracy_labels=None,
    confusion_matrices=None,
    class_names=None,
    base_dir="results",
    learning_rate=None
):
    # 1. Create main result directory
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)

    # 2. Create timestamped folder
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    result_path = os.path.join(base_dir, timestamp)
    os.makedirs(result_path)

    # 3. Create training and test subfolders
    train_path = os.path.join(result_path, "training")
    test_path = os.path.join(result_path, "test")
    os.makedirs(train_path)
    os.makedirs(test_path)

    print(f"Saving all plots to: {result_path}")

    # ---------- helper: metrics_from_confusion_matrix ----------
    def metrics_from_confusion_matrix(cm, class_labels):
        cm = np.asarray(cm)
        n_classes = cm.shape[0]
        total = cm.sum()

        rows = []
        for i in range(n_classes):
            TP = cm[i, i]
            FN = cm[i, :].sum() - TP
            FP = cm[:, i].sum() - TP
            TN = total - TP - FN - FP

            precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            class_acc = TP / (TP + FN) if (TP + FN) > 0 else 0.0
            support = TP + FN

            rows.append({
                "class": class_labels[i],
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "class_accuracy": class_acc,
                "support": support
            })

        df = pd.DataFrame(rows)

        overall_acc = np.trace(cm) / total if total > 0 else 0.0
        macro_precision = df["precision"].mean()
        macro_recall = df["recall"].mean()
        macro_f1 = df["f1"].mean()

        overall_row = {
            "class": "OVERALL",
            "precision": macro_precision,
            "recall": macro_recall,
            "f1": macro_f1,
            "class_accuracy": overall_acc,
            "support": total
        }
        df = pd.concat([df, pd.DataFrame([overall_row])], ignore_index=True)
        return df

    # ---------- 4. Training curves all folds ----------
    plt.figure(figsize=(10, 4))
    ax_loss = plt.subplot(1, 2, 1)
    ax_acc = plt.subplot(1, 2, 2)

    for fold_idx in sorted(train_accuracies.keys()):
        accs = train_accuracies[fold_idx]
        losses = train_losses[fold_idx]
        epochs = list(range(1, len(accs) + 1))

        ax_loss.plot(epochs, losses, marker='o', label=f"Fold {fold_idx + 1}")
        ax_acc.plot(epochs, accs, marker='o', label=f"Fold {fold_idx + 1}")

    ax_loss.set_title("Training Loss per Epoch (All Folds)")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.grid(True)
    ax_loss.legend()

    ax_acc.set_title("Training Accuracy per Epoch (All Folds)")
    ax_acc.set_xlabel("Epoch")
    ax_acc.set_ylabel("Accuracy")
    ax_acc.grid(True)
    ax_acc.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(train_path, "training_curves_all_folds.png"))
    plt.close()

    # ---------- 4b. Learning rate plot ----------
    if learning_rate is not None:
        plt.figure(figsize=(6, 4))

        # Case 1: dict of schedules per fold: {fold_idx: [lr1, lr2, ...], ...}
        if isinstance(learning_rate, dict):
            for fold_idx in sorted(learning_rate.keys()):
                lrs = np.atleast_1d(learning_rate[fold_idx])
                epochs = np.arange(1, len(lrs) + 1)
                plt.plot(epochs, lrs, marker='o', label=f"Fold {fold_idx + 1}")
            plt.legend()

        else:
            # Make it at least 1D (handles scalar or list/tuple/np.array)
            lrs = np.atleast_1d(learning_rate)

            # Use number of epochs from first fold to span a constant lr if needed
            first_fold = sorted(train_accuracies.keys())[0]
            n_epochs = len(train_accuracies[first_fold])

            # Case 2: single scalar learning rate -> constant schedule
            if len(lrs) == 1:
                plt.plot(
                    range(1, n_epochs + 1),
                    [float(lrs[0])] * n_epochs,
                    marker='o'
                )
            # Case 3: sequence of learning rates -> schedule over epochs
            else:
                epochs = np.arange(1, len(lrs) + 1)
                plt.plot(epochs, lrs, marker='o')

        plt.xlabel("Epoch")
        plt.ylabel("Learning rate")
        plt.title("Learning Rate Schedule")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(train_path, "learning_rate_schedule.png"))
        plt.close()

    # ----------------------------------------------
    # 5. Test accuracy histogram
    # ----------------------------------------------
    fold_labels = []
    fold_values = []

    for i in test_accuracy_labels:
        fold_labels.append(i+1)

    for set_idx in sorted(test_accuracies.keys()):
        value = test_accuracies[set_idx]
        if isinstance(value, (list, tuple, np.ndarray)):
            scalar = value[0] if len(value) == 1 else float(np.mean(value))
        else:
            scalar = value
        fold_values.append(float(scalar))

    plt.figure(figsize=(7, 5))
    x_pos = np.arange(len(fold_values))
    plt.bar(x_pos, fold_values)
    plt.title("Test Accuracy per Fold")
    plt.xlabel("Fold")
    plt.ylabel("Accuracy")
    plt.xticks(x_pos, fold_labels, rotation=45, ha="right")
    plt.grid(axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(test_path, "test_accuracy_histogram.png"))
    plt.close()

    # ---------- 6. Boxplot ----------
    plt.figure(figsize=(6, 5))
    plt.boxplot(fold_values, vert=True)
    plt.title("Distribution of Test Accuracies Across Folds")
    plt.ylabel("Accuracy")
    plt.xticks([1], ["Test Accuracies"])
    plt.grid(axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(test_path, "test_accuracy_boxplot.png"))
    plt.close()

    # ---------- 7. Confusion matrices + metrics ----------
    if confusion_matrices is not None:
        cm_path = os.path.join(test_path, "confusion_matrices")
        os.makedirs(cm_path, exist_ok=True)

        if class_names is not None:
            class_labels = list(class_names)
        else:
            first_cm = np.asarray(next(iter(confusion_matrices.values())))
            class_labels = [str(i) for i in range(first_cm.shape[0])]

        for fold_idx in sorted(confusion_matrices.keys()):
            cm = np.array(confusion_matrices[fold_idx])

            plt.figure(figsize=(6, 5))
            im = plt.imshow(cm, interpolation='nearest')
            plt.title(f"Confusion Matrix - Fold {fold_idx + 1}")
            plt.colorbar(im)
            plt.xlabel("Predicted")
            plt.ylabel("True")

            plt.xticks(np.arange(len(class_labels)), class_labels, rotation=45, ha="right")
            plt.yticks(np.arange(len(class_labels)), class_labels)

            thresh = cm.max() / 2 if cm.size > 0 else 0
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    plt.text(j, i, str(cm[i, j]),
                             ha="center", va="center",
                             color="white" if cm[i, j] > thresh else "black")

            plt.tight_layout()
            plt.savefig(os.path.join(cm_path, f"confusion_matrix_fold_{fold_idx+1}.png"))
            plt.close()

            df_metrics = metrics_from_confusion_matrix(cm, class_labels)
            df_metrics.to_csv(
                os.path.join(cm_path, f"class_metrics_fold_{fold_idx+1}.csv"),
                index=False
            )

        combined_cm = np.sum(np.array(list(confusion_matrices.values())), axis=0)

        plt.figure(figsize=(7, 6))
        im = plt.imshow(cm, interpolation='nearest', cmap='YlOrBr')
        plt.title("Combined Confusion Matrix (All Folds)")
        plt.colorbar(im)
        plt.xlabel("Predicted")
        plt.ylabel("True")

        plt.xticks(np.arange(len(class_labels)), class_labels, rotation=45, ha="right")
        plt.yticks(np.arange(len(class_labels)), class_labels)

        thresh = combined_cm.max() / 2 if combined_cm.size > 0 else 0
        for i in range(combined_cm.shape[0]):
            for j in range(combined_cm.shape[1]):
                plt.text(j, i, str(combined_cm[i, j]),
                         ha="center", va="center",
                         color="white" if combined_cm[i, j] > thresh else "black")

        plt.tight_layout()
        plt.savefig(os.path.join(cm_path, "combined_confusion_matrix.png"))
        plt.close()

        df_combined_metrics = metrics_from_confusion_matrix(combined_cm, class_labels)
        df_combined_metrics.to_csv(
            os.path.join(cm_path, "class_metrics_combined.csv"),
            index=False
        )

    print("Plots and metric tables saved successfully.")
    return result_path


### Training & Test

In [11]:
number_of_test_folds = 1
sets = KDHandler.create_splits(number_of_test_folds)
models = []
num_epochs = 100

training_losses_all = {}
training_accuracies_all = {}
test_accuracies_all = {}
test_accuracies_all_fold = {}
confusion_matrices_all = {}
lrs = {}
   


for i, combination_set in enumerate(sets):

    # Create the model and put it on the GPU if available
    new_model = AudioClassifier()
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    new_model = new_model.to(device)
    # Check that it is on Cuda
    next(new_model.parameters()).device

    print(f"##### Set {i} ####\n")

    print(f" - Starting training for set #{i}\n")
    train_set =[]
    for fold in combination_set[1]:
        if len(train_set) == 0:
            train_set = preprocessed_datasets[fold]
        else:
            train_set = train_set + preprocessed_datasets[fold]
 
    print(f"Training from the folds {combination_set[1]}\n")
    train_dl = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=False, pin_memory=True, num_workers=6, persistent_workers=True)
    losses, accuracies, lr = training(new_model, train_dl, num_epochs, device)
    lrs[i] = lr
    training_losses_all[i] = losses
    training_accuracies_all[i] = accuracies
    
    print(f"Starting testing for set #{i}\n")
    new_test_acc = []
    new_test_acc_fold = []   
    for fold in combination_set[0]:
        print(f"Testing on fold #{fold}")
        test_dl = torch.utils.data.DataLoader(preprocessed_datasets[fold], batch_size=128, shuffle=False)  
        acc, fold, all_preds = test(new_model, test_dl)
        cm = confusion_matrix(torch.cat(fold), all_preds)
        new_test_acc.append(acc)
        new_test_acc_fold.append(fold)
        confusion_matrices_all[i] = cm
    test_accuracies_all[i] = new_test_acc
    test_accuracies_all_fold[i] = new_test_acc_fold

    models.append(new_model)
    

##### Set 0 ####

 - Starting training for set #0

Training from the folds ['fold2', 'fold3', 'fold4', 'fold5', 'fold6', 'fold7', 'fold8', 'fold9', 'fold10']

Epoch: 0, Loss: 1.80, Accuracy: 0.39
Epoch: 1, Loss: 1.58, Accuracy: 0.45
Epoch: 2, Loss: 1.44, Accuracy: 0.51
Epoch: 3, Loss: 1.39, Accuracy: 0.53
Epoch: 4, Loss: 1.33, Accuracy: 0.55
Epoch: 5, Loss: 1.19, Accuracy: 0.60
Epoch: 6, Loss: 1.08, Accuracy: 0.64
Epoch: 7, Loss: 1.02, Accuracy: 0.66
Epoch: 8, Loss: 0.92, Accuracy: 0.69
Epoch: 9, Loss: 0.84, Accuracy: 0.72
Epoch: 10, Loss: 0.76, Accuracy: 0.75
Epoch: 11, Loss: 0.67, Accuracy: 0.78
Epoch: 12, Loss: 0.62, Accuracy: 0.80
Epoch: 13, Loss: 0.56, Accuracy: 0.81
Epoch: 14, Loss: 0.49, Accuracy: 0.84
Epoch: 15, Loss: 0.52, Accuracy: 0.83
Epoch: 16, Loss: 0.50, Accuracy: 0.84
Epoch: 17, Loss: 0.48, Accuracy: 0.85
Epoch: 18, Loss: 0.46, Accuracy: 0.86
Epoch: 19, Loss: 0.47, Accuracy: 0.85
Epoch: 20, Loss: 0.52, Accuracy: 0.85
Epoch: 21, Loss: 0.39, Accuracy: 0.88
Epoch: 22, Loss

### Results

In [12]:
result_dir = plot_and_save_cv_curves(
    train_accuracies=training_accuracies_all,
    train_losses=training_losses_all,
    test_accuracies=test_accuracies_all,
    test_accuracy_labels=test_accuracies_all_fold,
    confusion_matrices=confusion_matrices_all,
    class_names=None,                           
    base_dir="results",
    learning_rate=lrs
)


Saving all plots to: results/2025-11-29_22-03-39
Plots and metric tables saved successfully.
